<a href="https://colab.research.google.com/github/FaridRash/brain-ct-hemorrhage-segmentation/blob/main/Notebooks/Preprocess_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Google Drive mounting

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Unzip zipped image and mask folders

In [2]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/brain_ct_project/processed_png.zip"
extract_path = "/content/processed_data"

# create folder if not exists
os.makedirs(extract_path, exist_ok=True)

# unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully to:", extract_path)

Unzipped successfully to: /content/processed_data


In [3]:
print(os.listdir("/content/processed_data"))
print(len(os.listdir("/content/processed_data/images")))
print(len(os.listdir("/content/processed_data/masks")))

['images', 'masks']
2814
2814


#Convert Segmentation to Classification

In [4]:
import os
import cv2
import numpy as np
import pandas as pd

mask_dir = "/content/processed_data/masks"
image_dir = "/content/processed_data/images"

data = []

for mask_name in sorted(os.listdir(mask_dir)):

    mask_path = os.path.join(mask_dir, mask_name)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # check if any hemorrhage exists
    label = 1 if np.any(mask == 255) else 0

    # corresponding image name (same name)
    image_name = mask_name

    data.append([image_name, label])

# create dataframe
df = pd.DataFrame(data, columns=["filename", "label"])

# save CSV
csv_path = "/content/processed_data/labels.csv"
df.to_csv(csv_path, index=False)

print("Saved labels to:", csv_path)
print(df["label"].value_counts())

Saved labels to: /content/processed_data/labels.csv
label
0    2496
1     318
Name: count, dtype: int64


#Save the final dataset to Google Drive

In [5]:
import os
import shutil

source_base = "/content/drive/MyDrive/brain_ct_project/processed_png"
final_base = "/content/processed_png_final"

# remove if exists (to avoid conflicts)
if os.path.exists(final_base):
    shutil.rmtree(final_base)

# create new folder
os.makedirs(final_base, exist_ok=True)

print("Step 1 done.")

Step 1 done.


In [6]:
# update source path
source_base = "/content/processed_data"

# copy images
shutil.copytree(
    os.path.join(source_base, "images"),
    os.path.join(final_base, "images")
)

# copy labels.csv
shutil.copy(
    os.path.join(source_base, "labels.csv"),
    os.path.join(final_base, "labels.csv")
)

print("Step 2 done.")

Step 2 done.


In [7]:
import shutil

shutil.make_archive("/content/processed_png_final", 'zip', final_base)

print("Step 3 done — ZIP created.")

Step 3 done — ZIP created.


In [8]:
shutil.move(
    "/content/processed_png_final.zip",
    "/content/drive/MyDrive/brain_ct_project/processed_png_final.zip"
)

print("Step 4 done — saved in Google Drive.")

Step 4 done — saved in Google Drive.


In [9]:
import os

print(os.path.exists("/content/drive/MyDrive/brain_ct_project/processed_png_final.zip"))

True
